In [4]:
#!/usr/bin/env python
"""
Tunnel analysis Workflow

This script implements a workflow where you supply a receptor PDB file and a ligand file.
It creates a working subdirectory (named after the receptor), moves the PDB there,
modifies the CAVER configuration on the fly (adjusting first_frame, last_frame, and time_sparsity),
runs CAVER on the copied PDB (using the workdir as the -pdb option), and then for each tunnel found,
starts a separate CaverDock run using pyCaverDock.

Dependencies:
  - pyCaverDock (which provides classes like CaverDock, Receptor, Ligand, load_tunnel, discretize_tunnel, etc.)
  - pandas, matplotlib, py3Dmol, MDAnalysis
  - openbabel (via pybel)
"""

import os
import shutil
import subprocess
import tempfile
import pandas as pd
import matplotlib.pyplot as plt
import py3Dmol
from openbabel import pybel
from openbabel import openbabel as ob
import MDAnalysis as mda
import numpy as np
from scipy.spatial import Voronoi, ConvexHull, KDTree, Delaunay
from utils import utils

# Import pyCaverDock API
from pycaverdock import (
    CaverDock,
    Receptor,
    Ligand,
    load_tunnel,
    discretize_tunnel,
    Direction,
    TrajectoryType,
    EnergyProfilePlot,
    plot_results
)
from pycaverdock.experiment import convert_eprofile_analysis
from pycaverdock.utils import get_basename

def prepare_protein(input_file):
    """Convert a PDB receptor to PDBQT using OpenBabel."""
    if os.path.splitext(input_file)[1].lower() != ".pdbqt":
        base = os.path.splitext(input_file)[0]
        output_file = base + ".pdbqt"
        conv = ob.OBConversion()
        conv.SetInFormat("pdb")
        conv.SetOutFormat("pdbqt")
        conv.AddOption("r", ob.OBConversion.OUTOPTIONS)  # rigid output
        obmol = ob.OBMol()
        conv.ReadFile(obmol, input_file)
        obmol.CorrectForPH()  # optional
        obmol.AddHydrogens()
        conv.WriteFile(obmol, output_file)
        return output_file
    return input_file

def prepare_ligand(input_file):
    """Convert a ligand (SDF or MOL2) to PDBQT using OpenBabel."""
    ext = os.path.splitext(input_file)[1].lower()[1:]
    base = os.path.splitext(input_file)[0]
    output_file = base + ".pdbqt"
    mols = list(pybel.readfile(ext, input_file))
    if not mols:
        raise ValueError(f"No molecules found in {input_file}")
    mol = mols[0]
    mol.addh()
    mol.make3D()
    mol.calccharges(model='gasteiger')
    mol.write("pdbqt", output_file, overwrite=True)
    return output_file

class Miner:

    # Standard van der Waals radii for common atoms (in Angstroms)
    VDW_RADII = {
        'H': 1.20, 'C': 1.70, 'N': 1.55, 'O': 1.52, 'S': 1.80, 'P': 1.80,
        'F': 1.47, 'CL': 1.75, 'BR': 1.85, 'I': 1.98, 'NA': 2.27, 'MG': 1.73,
        'K': 2.75, 'CA': 2.31, 'ZN': 1.39, 'FE': 1.80, 'DEFAULT': 1.70
    }

    def __init__(self, universe, caver_path=None):
        """
        universe = MDAnalysis universe
        caver_path = paths to the CAVER executables.
        If not provided, they are assumed to be in subdirectories "caver" and "caverdock".
        """
        self.universe = universe

        if caver_path:
            self.caver_path = caver_path  
        else:
            try:
                self.caver_path = script_path = os.path.abspath(__file__)
            except:
                self.caver_path = "caver/"

    def internal_pocket_detection(self, protein_selection):
        """
        Simple internal pocket detection algorithm based on alpha spheres.
        This is a simplified version of fpocket's methodology.
        """
        pockets = []
        protein_positions = protein_selection.positions
        protein_radii = np.array([self.VDW_RADII.get(atom.element.upper(), self.VDW_RADII['DEFAULT']) 
                                 for atom in protein_selection])
        
        # Use KDTree for faster nearest neighbor searches
        protein_tree = KDTree(protein_positions)
        
        # Use the Voronoi diagram to find potential alpha sphere centers
        vor = Voronoi(protein_positions)
        
        # Vectorized operations for alpha sphere detection
        alpha_spheres = []
        batch_size = 1000  # Process vertices in batches for memory efficiency
        
        for i in range(0, len(vor.vertices), batch_size):
            batch_vertices = vor.vertices[i:i+batch_size]
            
            # Find distances from vertices to atoms using KDTree for speed
            distances, indices = protein_tree.query(batch_vertices, k=4)
            
            # Calculate minimum distances considering atom radii
            min_distances = distances - protein_radii[indices]
            min_dist_per_vertex = np.min(min_distances, axis=1)
            max_dist_per_vertex = np.max(distances, axis=1)
            
            # Find vertices that qualify as alpha spheres
            valid_mask = (min_dist_per_vertex > 0) & (min_dist_per_vertex < 3.0) & (max_dist_per_vertex < 10.0)
            valid_indices = np.where(valid_mask)[0]
            
            for j in valid_indices:
                alpha_spheres.append({
                    'center': batch_vertices[j],
                    'radius': min_dist_per_vertex[j],
                    'nearest_atoms': indices[j]
                })
        
        # Step 2: Cluster alpha spheres into pockets
        if not alpha_spheres:
            return pockets
        
        # Faster clustering using KDTree
        if len(alpha_spheres) > 0:
            centers = np.array([sphere['center'] for sphere in alpha_spheres])
            center_tree = KDTree(centers)
            
            # Find clusters using radius neighbor query
            clusters = []
            already_clustered = set()
            
            for i, sphere in enumerate(alpha_spheres):
                if i in already_clustered:
                    continue
                    
                # Find all neighbors within clustering distance
                neighbors = center_tree.query_ball_point(sphere['center'], 3.0)
                
                # Create a new cluster
                new_cluster = [sphere]
                already_clustered.add(i)
                
                # Add neighbors to cluster
                for neighbor_idx in neighbors:
                    if neighbor_idx != i and neighbor_idx not in already_clustered:
                        new_cluster.append(alpha_spheres[neighbor_idx])
                        already_clustered.add(neighbor_idx)
                
                clusters.append(new_cluster)
        
        # Step 3: Convert clusters to pockets
        for cluster in clusters:
            if len(cluster) >= 3:  # Minimum alpha spheres to define a pocket
                centers = [sphere['center'] for sphere in cluster]
                pockets.append({
                    'center': np.mean(centers, axis=0),
                    'alpha_spheres': centers,
                    'size': len(centers)
                })
        
        return pockets

    def set_start_point_from_coordinates(self, coordinates):
        """
        Set the tunnel starting point using specified coordinates.
        
        Parameters:
        -----------
        coordinates : array-like
            3D coordinates [x, y, z] of the starting point
        """
        self.start_point = np.array(coordinates)
        return self.start_point
    
    def set_start_point_from_selection(self, selection_string):
        """
        Set the tunnel starting point as the center of a MDAnalysis selection.
        
        Parameters:
        -----------
        selection_string : str
            MDAnalysis selection string (e.g., 'resid 100' or 'name CA and resid 50-60')
        """
        selection = self.universe.select_atoms(selection_string)
        
        if len(selection) == 0:
            raise ValueError(f"Selection '{selection_string}' did not match any atoms")
            
        self.start_point = selection.center_of_mass()
        return self.start_point
    
    def set_start_point_from_pocket(self, selection_string, max_pocket_distance=5.0):
        """
        Set the tunnel starting point as the center of a pocket containing a specified selection.
        Uses fpocket or similar methodology to identify pockets.
        
        Parameters:
        -----------
        ligand_selection : str
            MDAnalysis selection string for ligand or region of interest
        max_pocket_distance : float
            Maximum distance (Å) between selection and pocket center to consider it "containing" the selection
            
        Returns:
        --------
        numpy.ndarray
            Coordinates of the pocket center
        """
        # Identify the ligand/selection of interest
        selection = self.universe.select_atoms(selection_string)
        
        if len(selection) == 0:
            raise ValueError(f"Selection '{selection_string}' did not match any atoms")
        
        selection_center = selection.center_of_mass()
        protein_selection = self.universe.select_atoms("protein")

        # Implementation of a simplified pocket detection algorithm
        # This is inspired by fpocket but implemented directly to avoid external dependencies
        pockets = self.internal_pocket_detection(protein_selection)
        
        # Find the closest pocket to the selection center
        closest_pocket = None
        min_distance = float('inf')
        
        for pocket in pockets:
            distance = np.linalg.norm(pocket['center'] - selection_center)
            if distance < min_distance and distance <= max_pocket_distance:
                min_distance = distance
                closest_pocket = pocket
        
        if closest_pocket is None:
            raise ValueError(f"No pocket found containing the selection '{selection_string}'")
        
        self.start_point = closest_pocket['center']
        return self.start_point

    
    def run_caver(self, receptor_file, first_frame=1, last_frame=1, time_sparsity=1):
        """
        Run CAVER on the receptor PDB using a temporary directory in the original working directory.

        This method:
        • Creates a temporary working directory in the script's original working directory.
        • Copies the receptor PDB into that directory.
        • Reads and modifies the default CAVER config file to set first_frame, last_frame, and time_sparsity.
        • Runs CAVER with -pdb set to the working directory.
        • Cleans up the temporary directory after execution.
        """
        # Get the directory where the script was originally run
        original_cwd = os.getcwd()
        
        # Create a temporary directory within the original working directory
        temp_dir = tempfile.mkdtemp(dir=original_cwd)
        print(temp_dir)
        print(self.caver_path)
        
        try:
            # Copy the receptor file into temp_dir
            receptor_path = os.path.join(temp_dir, os.path.basename(receptor_file))
            shutil.copy(receptor_file, temp_dir)

            # Read the default config file (assumed to be in the same dir as caver.jar)
            default_config_path = f"{self.caver_path}/config_default.txt"
            with open(default_config_path, "r") as f:
                config_lines = f.readlines()

            starting_point_set = False

            # Modify parameters: first_frame, last_frame, time_sparsity
            new_config_lines = []
            for line in config_lines:
                stripped = line.strip()
                if stripped.startswith("time_sparsity"):
                    new_config_lines.append(f"time_sparsity {time_sparsity}\n")
                elif stripped.startswith("first_frame"):
                    new_config_lines.append(f"first_frame {first_frame}\n")
                elif stripped.startswith("last_frame"):
                    new_config_lines.append(f"last_frame {last_frame}\n")
                elif stripped.startswith("starting_point"):
                    if starting_point_set == True:
                        pass
                    elif starting_point_set == False:
                        new_config_lines.append(f"starting_point_coordinates {utils.numpy_to_blankspace_sep_str(self.start_point)}\n")

                else:
                    new_config_lines.append(line)

            # Save modified config file in temp_dir
            new_config_path = os.path.join(temp_dir, "config.txt")
            with open(new_config_path, "w") as f:
                f.writelines(new_config_lines)

            # Build the CAVER command
            cmd = [
                "java", "-Xmx1200m", 
                "-jar", f"{self.caver_path}/caver.jar",
                "-home", f"{self.caver_path}/",
                "-pdb", f"{temp_dir}/",
                "-conf", new_config_path,
                "-out", f"{temp_dir}/"
            ]
            print("Running CAVER:", " ".join(cmd))
            subprocess.check_call(cmd)

            # Assume tunnels are saved as "tunnels.pdb" in temp_dir
            tunnels_file = os.path.join(temp_dir, "tunnels.pdb")
            return tunnels_file, temp_dir

        finally:
            #Clean up the temporary directory after execution
            #shutil.rmtree(temp_dir, ignore_errors=True)
            print("hi")

    def visualize_tunnel(self, receptor_file, tunnel_file, output_html):
        view = py3Dmol.view(width=800, height=600)
        with open(receptor_file, "r") as f:
            pdb_data = f.read()
        view.addModel(pdb_data, "pdb")
        with open(tunnel_file, "r") as f:
            tunnel_data = f.read()
        view.addModel(tunnel_data, "pdb")
        view.setStyle({'model': 0}, {'cartoon': {'color': 'spectrum'}})
        view.setStyle({'model': 1}, {'stick': {'color': 'red', 'radius': 0.3}})
        view.zoomTo()
        html_str = view._make_html()
        with open(output_html, "w") as f:
            f.write(html_str)
        print(f"Interactive visualization HTML saved to {output_html}")

if __name__ == "__main__":
    receptor_pdb = "1be0.pdb"    # Supply your receptor PDB file
    ligand_file = "EtOH.sdf"      # Supply your ligand file (SDF or MOL2)
    
    universe = mda.Universe(receptor_pdb)

    miner = Miner(universe, caver_path="caver")

    #set start point
    miner.set_start_point_from_pocket("resid 124 or resid 226 or resid 175")
    # Adjust first_frame, last_frame, and time_sparsity as desired.
    miner.run_caver("1be0.pdb")


/home/erikna/compchem/WhatCat/miner/tmpnkq6ftw1
caver
Running CAVER: java -Xmx1200m -jar caver/caver.jar -home caver/ -pdb /home/erikna/compchem/WhatCat/miner/tmpnkq6ftw1/ -conf /home/erikna/compchem/WhatCat/miner/tmpnkq6ftw1/config.txt -out /home/erikna/compchem/WhatCat/miner/tmpnkq6ftw1/
Caver computation started.
Merging all PDB files into one multimodel PDB file.
Settings loaded from:
/home/erikna/compchem/WhatCat/miner/tmpnkq6ftw1/config.txt

Basic parameters:
starting_point_coordinates [25.994804300307816, 31.300555483650815, 30.739844306367118]
probe_radius 0.9
shell_radius 3.0
shell_depth 4.0
frame_weighting_coefficient 1.0
frame_clustering_threshold 1.0

*** Processing 1be0.pdb ***
0 redundant tunnels removed.
0 tunnels stored.

Loading tunnels from /home/erikna/compchem/WhatCat/miner/tmpnkq6ftw1/data/tunnels/tunnels_1be0.pdb.obj
Loaded 0 tunnels for 1be0.pdb.
0 tunnels successfully loaded.

Leaving only the cheapest tunnel per cluster per snapshot - 0 tunnels kept.
Statistica

Mar 11, 2025 8:17:00 AM caver.ui.Launcher cluster
Mar 11, 2025 8:17:00 AM caver.ui.Launcher run
SEVERE: Unexpected.
java.lang.NullPointerException: Cannot invoke "geometry.primitives.Point.distance(geometry.primitives.Point)" because "averageVoronoiOrigin" is null
	at caver.ui.Statistics.saveResidueClusterTouches(Statistics.java:475)
	at caver.ui.Launcher.saveStatistics(Launcher.java:1026)
	at caver.ui.Launcher.output(Launcher.java:813)
	at caver.ui.Launcher.compute(Launcher.java:1222)
	at caver.ui.Launcher.run(Launcher.java:1276)
	at caver.ui.Launcher.main(Launcher.java:1411)



In [ ]:
#TODO error is missing start coordinates. these need to be written to the caver config file
#TODO fix the paths in the caver call. these are not correct.

NameError: name '__file__' is not defined